# AQDrop Job Submission and Retrieval Example

This notebook demonstrates how to submit a simple Bell-state quantum circuit to an AQDrop queue and then retrieve the results.

In [23]:
# IMPORTS
import os
from qiskit import QuantumCircuit
from aqdrop import AqdropClient
from aqdrop.cli_utils import connect_verbose, print_job_table

## Initialize Client

In [13]:
# note that AqdropClient automatically instantiates from the environment variables:
# AQDROP_USERNAME, NERSC_OIDC_TOKEN, AQDROP_HOSTNAME
client: AqdropClient = connect_verbose()

Connecting...
Connected to AQDROP service as user evan_u.



## Initialize Circuit

We test a circuit that makes a Bell state. This circuit is a simple entanglement between qubits 0 and 1.

In [15]:
def circ_bell():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure(0, 0)
    qc.measure(1, 1)
    return qc

qc = circ_bell()
print("Bell State Circuit:")
print(qc.draw())

Bell State Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [27]:
# Assemble job input
circuits = [qc]
job_meta = {
    "shots": [SHOTS], 
    "comment": "Notebook Bell State Job", 
    "queue_name": QUEUE_NAME, 
    "pref_qubits": None
}

submitted = client.submit_qiskit(QUEUE_NAME, circuits, job_meta)

print(f"Job submitted successfully! Job ID: {submitted['id']}")
print()
print("Submitted job details:")
print_job_table([{k: v for k, v in submitted.items() if k != 'input'}])

Job submitted successfully! Job ID: 1090

Submitted job details:
  id   owner_name   queue_name   status   output   last_action
  --   ----------   ----------   ------   ------   -----------
1090   evan_u       X6Y3         queued   None     2026-05-06 23:02:43 PDT


## Retrieving the Job

Now we will retrieve the results for the job we just submitted. Note that depending on the queue, the job might still be `queued`.

In [10]:
# Retrieve the job using the ID from the previous step
job = user.pull_job(job_id)
print(f"Job Status: {job['status']}")

pulled job: {'id': 1082, 'owner_name': 'evan_u', 'queue_name': 'X6Y3', 'status': 'queued'}
Job Status: queued


In [11]:
# Parse the job into its components
circL, inputMD, output, transpiledL = user.parse_job()

if job['status'] == "success":
    user.print_shot_summary()
    print(f"Total Execution Time: {output['tot_exec_time']:.1f} sec")
    print(f"Received Total Shots: {output['tot_shots']}")
    print(f"Calibration Version: {output['calib_ver']}")
    print(f"Execution Date: {output['exec_date']}")
    
    if VERBOSITY > 0:
        user.print_output_counts()
else:
    print("Results are not available yet or the job failed.")

parsed job: 1 input circuits, no output yet (status=queued)
Results are not available yet or the job failed.


In [6]:
# Display circuits and transpiled circuits
print(f"Packed circuits: {len(circL)}, total requested shots: {sum(inputMD['shots'])}")

if VERBOSITY > 1:
    for idx, qc in enumerate(circL):
        print(f"\nCircuit {idx}:")
        print(qc)
    if transpiledL:
        for idx, qc in enumerate(transpiledL):
            print(f"\nTranspiled Circuit {idx}:")
            print(qc)

Packed circuits: 1, total requested shots: 4000
